# plotmux — case study: reproducing bokeh's slope example

This notebook reproduces [bokeh's `slope` annotation example](https://docs.bokeh.org/en/latest/docs/examples/basic/annotations/slope.html) -- scatter markers with a separate yellow fill / black edge drawn with `alpha=0.8`, plus a dashed reference line of gradient 2 and y-intercept 10 at `line_width=4`, on a figure with a light-gray background and `y_range.start = 0` -- through plotmux's unified API, unchanged, on all four backends.

See `DESIGN.md`, section 8.1, for the full case study writeup. In short: `plotmux.slope(...)` is a standalone spec only on matplotlib and bokeh (the two backends with a native "line by slope, independent of data range" primitive); on altair and xy, `SlopeSpec` is only supported as a `layer()` child alongside a data-bound sibling, which derives the line's endpoints from that sibling's data range. The scatter + slope combination below already has that data-bound sibling, so the *same* `plotmux.layer(...)` call reproduces the example unchanged on all four backends.

In [ ]:
import numpy as np

import plotmux
from plotmux.figure import Figure
from plotmux.specs import ScatterSpec, SlopeSpec

# Bokeh renders lazily: its native figure's `_repr_html_` only shows a plain
# text description of the model until `output_notebook()` registers Bokeh's
# own IPython display hook (BokehJS), which is what actually draws the plot
# inline. This is harmless to call even for notebook runs that never pick
# the bokeh backend.
try:
    from bokeh.io import output_notebook

    output_notebook()
except ImportError:
    pass

## Reproducing bokeh's own generated data

Same equation (`gradient=2, intercept=10`), same `x` range, and the same additive Gaussian noise bokeh's own example uses to scatter the points around the line.

In [ ]:
rng = np.random.default_rng(42)

gradient, intercept = 2, 10
x = np.arange(0, 20, 0.2)
y = gradient * x + intercept + rng.normal(0, 4, size=x.shape)

## Same call, every backend

The loop below reruns the *exact same* `plotmux.layer(...)` call across all four backends to confirm the reproduction is unchanged, not backend-specific code paths in disguise.

In [ ]:
def make_slope_figure(x: np.ndarray, y: np.ndarray, gradient: float, intercept: float) -> Figure:
    r"""Build the scatter + slope figure reproducing bokeh's example."""
    return plotmux.layer(
        ScatterSpec(
            x=x,
            y=y,
            color="yellow",
            edgecolor="black",
            size=8,
            alpha=0.8,
        ),
        SlopeSpec(
            gradient=gradient,
            intercept=intercept,
            color="#5386a3",
            linestyle="dashed",
            linewidth=4,
        ),
        background_color="#fafafa",
        ymin=0,
    )

### matplotlib

In [ ]:
with plotmux.backend("matplotlib"):
    fig_mpl = make_slope_figure(x, y, gradient, intercept)
fig_mpl.to_native()

### bokeh

In [ ]:
from bokeh.io import show

with plotmux.backend("bokeh"):
    fig_bokeh = make_slope_figure(x, y, gradient, intercept)
show(fig_bokeh.to_native())

### altair

Supported here only because `SlopeSpec` has a data-bound `ScatterSpec` sibling in the same `layer()` call to derive its endpoints' x-range from -- a standalone `plotmux.slope(...)` still raises `UnsupportedSpecError` on altair (see the next section).

In [ ]:
with plotmux.backend("altair"):
    fig_altair = make_slope_figure(x, y, gradient, intercept)
fig_altair.to_native()

### xy

Same caveat as altair above.

In [ ]:
with plotmux.backend("xy"):
    fig_xy = make_slope_figure(x, y, gradient, intercept)
fig_xy.to_native()

## The altair/xy caveat: a standalone `slope()` has no data to derive a range from

A standalone `plotmux.slope(...)` (no data-bound `layer()` sibling) works on matplotlib and bokeh, since both have a native slope-by-itself primitive. On altair and xy it raises `UnsupportedSpecError`, since neither backend can draw a line from just a `(gradient, intercept)` pair -- there's no data to derive concrete endpoints from.

In [ ]:
import contextlib

import plotmux.exceptions

for backend_name in ["altair", "xy"]:
    with (
        contextlib.suppress(plotmux.exceptions.UnsupportedSpecError),
        plotmux.backend(backend_name),
    ):
        plotmux.slope(gradient, intercept)

matplotlib and bokeh, by contrast, render a standalone `plotmux.slope(...)` on its own, with no scatter sibling needed:

In [ ]:
with plotmux.backend("matplotlib"):
    fig_standalone = plotmux.slope(gradient, intercept, linestyle="dashed", linewidth=4)
fig_standalone.to_native()